## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

>  **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 5 dimulai**.
>  **Sifat tugas:** individu.

### Konteks / Skenario

Tim engineering platform e-commerce (skenario yang sama dari Pertemuan 2-3) resmi meminta seluruh proses analisis data yang sebelumnya memakai pandas **dipindahkan ke PySpark**, karena volume data transaksi diperkirakan akan tumbuh sangat besar dalam waktu dekat sehingga pandas (yang memuat semua data ke RAM) tidak lagi memadai. Sebagai data analyst yang baru belajar PySpark, anda ditugaskan membuktikan bahwa seluruh alur analisis dapat direplikasi menggunakan PySpark, **membaca data langsung dari HDFS**.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset baru (transaksi bulan September 2026, lebih banyak baris dari sebelumnya) dan mengunggahnya ke HDFS.

In [3]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


**Membuat SparkSession**

In [7]:
from pyspark.sql import SparkSession

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Tugas4") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


**A. Membaca dan Eksplorasi Awal** *(bobot 15%)*

Baca dataset dari HDFS, tampilkan `printSchema()`, jumlah baris (`count()`), dan 10 baris pertama (`show(10)`).

In [4]:
df_dari_hdfs = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True, inferSchema=True
)
df_dari_hdfs.printSchema()
print("Jumlah baris dari HDFS:", df_dari_hdfs.count())
df_dari_hdfs.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris dari HDFS: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semaran

**B. Menangani Data Kosong** *(bobot 15%)*

Kolom `rating` memiliki nilai kosong. Tampilkan berapa banyak, lalu gunakan `df.na.fill()` atau `df.na.drop()` (pilih salah satu, jelaskan alasannya pada markdown cell) untuk menanganinya.

In [15]:
from pyspark.sql.functions import col

print("Banyaknya kolom rating kosong sebelum: ",df_dari_hdfs.filter(col("rating").isNull()).count())

df_bersih = df_dari_hdfs.na.drop(subset=["rating"])
print("Banyaknya kolom rating kosong sesudah: ",df_bersih.filter(col("rating").isNull()).count())

Banyaknya kolom rating kosong sebelum:  204
Banyaknya kolom rating kosong sesudah:  0


Setelah melakukan pengecekan kolom kosong pada rating saya menemukan 204 baris rating yang berisi Null. Untuk menanganinya saya gunakan drop karena pada soal selanjutnya terdapat soal yang meminta kita untuk menampilkan rata-rata rating dari masing masing metode pembayaran, jika kita menggunakan fill akan merusak hasil rata rata yang sebenarnya walaupun kita mengorbankan sebanyak 204 baris rating.

**C. Transformasi Data** *(bobot 20%)*

Tambahkan kolom `total_pendapatan` (`unit_terjual x harga_satuan`), lalu tambahkan kolom `tier_transaksi` yang bernilai `"Besar"` jika `total_pendapatan > 500000`, atau `"Kecil"` jika sebaliknya 

In [27]:
from pyspark.sql.functions import when

df_bersih = df_bersih.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
df_bersih = df_bersih.withColumn("tier_transaksi", when(df_bersih["total_pendapatan"] > 500000, "Besar").otherwise("Kecil"))

df_bersih.show(10, truncate=False)

+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|tanggal            |kategori              |kota      |unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|Rumah Tangga          |Yogyakarta|3           |90000       |COD              |4.0   |270000          |Kecil         |
|ORD-3001|2026-09-04 00:00:00|Makanan & Minuman     |Solo      |3           |200000      |E-Wallet         |5.0   |600000          |Besar         |
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecantikan|Semarang  |8           |60000       |E-Wallet         |3.0   |480000          |Kecil         |
|ORD-3003|2026-09-09 00:00:00|Makanan & Minuman     |Semarang  |6           |350000      |Transfer Bank    |4.0 

**D. Analisis dengan GroupBy** *(bobot 30%)*

Jawablah dengan kode PySpark (bukan pandas):
1. Kategori apa yang memiliki `total_pendapatan` tertinggi?
2. Kota mana dengan jumlah transaksi **tier "Besar"** terbanyak?
3. Berapa rata-rata `rating` untuk masing-masing `metode_pembayaran` (data kosong sudah ditangani di bagian B)?

In [59]:
from pyspark.sql.functions import sum as spark_sum, count, avg

kategori = df_bersih.groupBy("kategori").agg(spark_sum("total_pendapatan").alias("total_pendapatan")).orderBy(col("total_pendapatan").desc())
kategori.show(1)

kota_besar = df_bersih.filter(df_bersih["tier_transaksi"] == "Besar").groupBy("kota").agg(count("order_id").alias("jumlah")).orderBy(col("jumlah").desc())
kota_besar.show(1)

rating_mp = df_bersih.groupBy("metode_pembayaran").agg(avg("rating").alias("rata_rating"))
rating_mp.show()

+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       108285000|
+------------+----------------+
only showing top 1 row

+----+------+
|kota|jumlah|
+----+------+
|Solo|    74|
+----+------+
only showing top 1 row

+-----------------+-----------------+
|metode_pembayaran|      rata_rating|
+-----------------+-----------------+
|              COD|4.172413793103448|
|    Transfer Bank| 4.16256157635468|
|     Kartu Kredit|4.109947643979058|
|         E-Wallet|4.135678391959799|
+-----------------+-----------------+



**E. Menyimpan Hasil ke HDFS** *(bobot 20%)*

Simpan DataFrame hasil olahan bagian C (lengkap dengan kolom `total_pendapatan` dan `tier_transaksi`) ke HDFS dalam format CSV baru, kemudian verifikasi apakah sudah berhasil.

In [64]:
df_bersih.write.mode("overwrite").csv("hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi_september", header=True)

!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_transaksi_september

Found 2 items
-rw-r--r--   3 asta supergroup          0 2026-09-16 14:18 /user/mahasiswa/tugas4/hasil_transaksi_september/_SUCCESS
-rw-r--r--   3 asta supergroup      73357 2026-09-16 14:18 /user/mahasiswa/tugas4/hasil_transaksi_september/part-00000-cf497ff3-16f1-4ea0-a659-96575e54e0a8-c000.csv


In [67]:
df_verifikasi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi_september",
    header=True, inferSchema=True
)
df_verifikasi.printSchema()
df_verifikasi.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- total_pendapatan: integer (nullable = true)
 |-- tier_transaksi: string (nullable = true)

+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|tanggal            |kategori              |kota      |unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|Rumah Tangga          |Yogyakarta|3           |90000       |COD              |4.0   |27000

Spark menyimpan hasilnya sebagai beberapa berkas partisi (part-00000, dst.) karena cara kerja Spark memang berbeda dengan pandas. Kalau pandas memproses semua data dalam satu proses dan satu memori (RAM) yang sama sehingga hasilnya bisa langsung ditulis jadi satu file utuh, Spark justru memecah DataFrame menjadi beberapa partisi yang dikerjakan secara paralel oleh executor yang berbeda. Saat proses penulisan ke HDFS berlangsung setiap executor menulis hasil partisinya masing-masing secara terpisah dan bersamaan, sehingga yang muncul bukan satu file tunggal melainkan banyak file kecil.  Pada kasus ini, hanya muncul satu file part-00000 karena ukuran datanya masih relatif kecil (sekitar 1000 baris dikurangi 204 baris yang di-drop) sehingga Spark cukup menempatkannya dalam satu partisi saja. Namun kalau datasetnya jauh lebih besar biasanya akan muncul banyak file seperti part-00000, part-00001, part-00002, dan seterusnya—jumlahnya tergantung berapa banyak partisi yang dibuat Spark untuk memproses data tersebut secara terdistribusi.

**Eksplorasi**

saya akan melakukan eksplorasi seberapa pengaruh penggunaan drop dan terakhir akan membandingkannya dengan fill

In [68]:
from pyspark.sql.functions import round as spark_round

#1
df_penuh = (df_dari_hdfs
            .withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
            .withColumn("tier_transaksi",
                        when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")))

#2
total_drop  = df_bersih.agg(spark_sum("total_pendapatan")).collect()[0][0]
total_penuh = df_penuh.agg(spark_sum("total_pendapatan")).collect()[0][0]
selisih = total_penuh - total_drop

print(f"Total pendapatan (drop) : Rp{total_drop:,}")
print(f"Total pendapatan (penuh): Rp{total_penuh:,}")
print(f"Selisih                 : Rp{selisih:,} ({selisih/total_penuh*100:.2f}% dari total)")

Total pendapatan (drop) : Rp597,690,000
Total pendapatan (penuh): Rp760,170,000
Selisih                 : Rp162,480,000 (21.37% dari total)


In [70]:
#3
print("(kategori tertinggi, data penuh)")
(df_penuh.groupBy("kategori").agg(spark_sum("total_pendapatan").alias("total_pendapatan")).orderBy(col("total_pendapatan").desc()).show())

print("(kota tier 'Besar' terbanyak, data penuh)")
(df_penuh.filter(col("tier_transaksi") == "Besar").groupBy("kota").count().orderBy(col("count").desc()).show())

(kategori tertinggi, data penuh)
+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+

(kota tier 'Besar' terbanyak, data penuh)
+----------+-----+
|      kota|count|
+----------+-----+
|      Solo|   92|
|  Magelang|   78|
|   Kebumen|   78|
|Yogyakarta|   75|
| Purworejo|   66|
|  Semarang|   65|
+----------+-----+



In [71]:
#4
rata_global = df_dari_hdfs.agg(avg("rating")).collect()[0][0]
df_fill = df_dari_hdfs.na.fill({"rating": rata_global})

hasil_drop = (df_bersih.groupBy("metode_pembayaran").agg(spark_round(avg("rating"), 4).alias("rating_drop")))
hasil_fill = (df_fill.groupBy("metode_pembayaran").agg(spark_round(avg("rating"), 4).alias("rating_fill")))

(hasil_drop.join(hasil_fill, on="metode_pembayaran").withColumn("selisih", spark_round(col("rating_fill") - col("rating_drop"), 4)).orderBy("metode_pembayaran").show())

print(f"Rata-rata rating global yang dipakai untuk fill: {rata_global:.4f}")

+-----------------+-----------+-----------+-------+
|metode_pembayaran|rating_drop|rating_fill|selisih|
+-----------------+-----------+-----------+-------+
|              COD|     4.1724|     4.1673|-0.0051|
|         E-Wallet|     4.1357|     4.1377|  0.002|
|     Kartu Kredit|     4.1099|     4.1179|  0.008|
|    Transfer Bank|     4.1626|     4.1592|-0.0034|
+-----------------+-----------+-----------+-------+

Rata-rata rating global yang dipakai untuk fill: 4.1457
